# Data Exploration

**Authors:** Benedikt Prisett & Stijn Diemel

Initial data exploration of the `simpsons_script_lines.csv` dataset to understand its structure, identify data quality issues and motivate the cleaning steps in [02_cleaning.ipynb](./02_cleaning.ipynb).


## 1. Setup & Overview


In [1]:
import pandas as pd

df = pd.read_csv("../data/raw_data/simpsons_script_lines.csv", low_memory=False)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDtypes:")
print(df.dtypes)

Shape: 158271 rows, 13 columns

Columns: ['id', 'episode_id', 'number', 'raw_text', 'timestamp_in_ms', 'speaking_line', 'character_id', 'location_id', 'raw_character_text', 'raw_location_text', 'spoken_words', 'normalized_text', 'word_count']

Dtypes:
id                      int64
episode_id              int64
number                  int64
raw_text               object
timestamp_in_ms        object
speaking_line          object
character_id           object
location_id           float64
raw_character_text     object
raw_location_text      object
spoken_words           object
normalized_text        object
word_count             object
dtype: object


In [2]:
df.sort_values("id").head(10)

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
148761,1,1,0,(Street: ext. street - establishing - night),8000,false,NaN,1.0,NaN,Street,NaN,NaN,NaN
148762,2,1,1,(Car: int. car - night),8000,false,NaN,2.0,NaN,Car,NaN,NaN,NaN
148763,3,1,2,"Marge Simpson: Ooo, careful, Homer.",8000,true,1,2.0,Marge Simpson,Car,"Ooo, careful, Homer.",ooo careful homer,3
148764,4,1,3,Homer Simpson: There's no time to be careful.,10000,true,2,2.0,Homer Simpson,Car,There's no time to be careful.,theres no time to be careful,6
148765,5,1,4,Homer Simpson: We're late.,10000,true,2,2.0,Homer Simpson,Car,We're late.,were late,2
148766,6,1,5,(Springfield Elementary School: Ext. springfie...,24000,false,NaN,3.0,NaN,Springfield Elementary School,NaN,NaN,NaN
148767,7,1,6,(Auditorium: int. auditorium - night),24000,false,NaN,4.0,NaN,Auditorium,NaN,NaN,NaN
148768,8,1,7,"Marge Simpson: (HUSHED VOICE) Sorry, Excuse us...",24000,true,1,4.0,Marge Simpson,Auditorium,"Sorry, Excuse us. Pardon me...",sorry excuse us pardon me,5
148769,9,1,8,"Homer Simpson: (SIMULTANEOUSLY) Hey, Norman. H...",26000,true,2,4.0,Homer Simpson,Auditorium,"Hey, Norman. How's it going? So you got dragge...",hey norman hows it going so you got dragged do...,21
148770,10,1,9,Homer Simpson: Pardon my galoshes. (CHUCKLES),34000,true,2,4.0,Homer Simpson,Auditorium,Pardon my galoshes.,pardon my galoshes,3


## 2. Missing Values


In [3]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(1)
pd.DataFrame({"nulls": nulls, "percent": null_pct})[nulls > 0]

,nulls,percent
character_id,17521,11.1
location_id,407,0.3
raw_character_text,17522,11.1
raw_location_text,408,0.3
spoken_words,26159,16.5
normalized_text,26184,16.5
word_count,26159,16.5


In [4]:
# Do rows with NaN spoken_words correspond to non-speaking lines?
nan_spoken = df[df["spoken_words"].isna()]
print(f"Rows with NaN spoken_words: {len(nan_spoken)}")
print(f"\nTheir speaking_line values:")
print(nan_spoken["speaking_line"].value_counts())
print(f"\nSample of non-speaking rows:")
nan_spoken[["raw_text", "speaking_line"]].head(5)

Rows with NaN spoken_words: 26159

Their speaking_line values:
speaking_line
false                                    26158
Guess what. I also play Frankenstein!        1
Name: count, dtype: int64

Sample of non-speaking rows:


,raw_text,speaking_line
8,(Apartment Building: Ext. apartment building -...,false
16,(Springfield Elementary School: EXT. ELEMENTAR...,false
27,Bart Simpson: (ANGUISHED SCREAM),false
29,(Moe's Tavern: Int. Moe's - evening),false
35,(Train Station: int. train station - afternoon),false


All rows with missing `spoken_words` are non-speaking lines (stage directions, scene descriptions). These will be filtered out in cleaning.


## 3. `speaking_line` Column Investigation


In [5]:
print("speaking_line value counts:")
print(df["speaking_line"].value_counts())

speaking_line value counts:
speaking_line
true                                     132112
false                                     26158
Guess what. I also play Frankenstein!         1
Name: count, dtype: int64


In [6]:
# There seems to be one corrupted row where dialogue leaked into speaking_line (value is neither "true" nor "false")
corrupted_sl = df[~df["speaking_line"].isin(["true", "false"])]
print(f"Corrupted speaking_line rows: {len(corrupted_sl)}")
corrupted_sl[["raw_text", "speaking_line", "word_count"]]

Corrupted speaking_line rows: 1


,raw_text,speaking_line,word_count
142024,"""ABRAHAM LINCOLN: (FURIOUS) Guess what. I also...",Guess what. I also play Frankenstein!,NaN


The `speaking_line` column is stored as strings (`"true"`/`"false"`), not booleans. One row has dialogue text in this field.


## 4. `word_count` Parsing Investigation

The `word_count` column has dtype `object` instead of numeric. Non-numeric values need to be investigated.


In [13]:
df["word_count_numeric"] = pd.to_numeric(
    df["word_count"], errors="coerce"
)  # This will convert non-numeric values to NaN
bad_wc = df[
    df["word_count"].notna() & df["word_count_numeric"].isna()
]  # Rows where original word_count is not NaN but numeric conversion failed
print(f"Rows with non-numeric word_count: {len(bad_wc)}")
print("\nTheir word_count values:")
print(bad_wc["word_count"].value_counts())
print("\nSample raw_text:")
bad_wc[["raw_text", "word_count"]].head(5)

Rows with non-numeric word_count: 15

Their word_count values:
word_count
true                                                                              10
I'm a Selma."                                                                      1
and presto -- you're part of the under-clown railroad! So you got any talent?"     1
just don't ask me to drive you to the airport."                                    1
the fact that it wasn't me. I've never felt so alive."                             1
we're all out                                                                      1
Name: count, dtype: int64

Sample raw_text:


,raw_text,word_count
8082,"Singers: (SINGING) ""IT'S THE FIRST ANNUAL MONT...",true
71799,"Revue Cast Members: ""WE'RE THE PERFORMERS YOU ...",true
81136,"Homer Simpson: ""I-M-O--",true
86744,"Patty Bouvier: ""Are you a 'Patty' or a 'Selma?...","I'm a Selma."""
115436,"Moe Szyslak: ""The reason I left you is simple:",true


15 rows have corrupted `word_count` values. Likely because the dialogue contains unescaped quote characters. These rows would need manual correction and will be dropped in cleaning as they are only a very small fraction of the total data.


## 4b. `word_count` Outliers

Beyond non-numeric values, we should check for numeric `word_count` values that are valid integers but clearly wrong.


In [20]:
print(df["word_count_numeric"].describe())
# print(f"\nRows with word_count > 200: {(df['word_count_numeric'] > 200).sum()}")

outliers = df.nlargest(15, "word_count_numeric")
print("\nOutlier rows:")
outliers[["id", "episode_id", "raw_character_text", "word_count_numeric", "raw_text"]]

count    1.320970e+05
mean     4.261253e+01
std      5.245308e+03
min      0.000000e+00
25%      4.000000e+00
50%      8.000000e+00
75%      1.300000e+01
max      1.154000e+06
Name: word_count_numeric, dtype: float64

Outlier rows:


,id,episode_id,raw_character_text,word_count_numeric,raw_text
153016,4263,14,Entire Town,1154000.0,"Entire Town: ""GONE AWAY IS THE BLUEBIRD / HERE..."
59908,69807,243,Marge Simpson,1145000.0,"Marge Simpson: (CONTINUING) ""My gold is in the..."
73537,83465,289,Robert Pinsky,672000.0,"Robert Pinsky: ""Impossible to tell in writing...."
52605,62483,220,ABBA,571000.0,"ABBA: ""If you wanna be my lover / You gotta ge..."
78951,88907,308,Homer Simpson,409000.0,"Homer Simpson: (READING) ""Nausea... cravings....."
77228,87170,302,Lisa Simpson,147000.0,"Lisa Simpson: (READING) ""Molochai desiratum ma..."
101152,111237,390,Lisa Simpson,117000.0,"Lisa Simpson: ""...Hitachee Tribe. (NERVOUS GIG..."
154163,5432,19,Bart Simpson,108000.0,"Bart Simpson: (WRITING) ""One o'clock -- still ..."
13085,22701,76,Grampa Simpson,122.0,Grampa Simpson: One trick is to tell them stor...
124004,134211,478,Randy Newman,116.0,"""Randy Newman"": It was not! Who else in here's..."


8 rows have word counts of 108,000 - 1,154,000. These are `timestamp_in_ms` values that leaked into the `word_count` column. The longest real lines in the show has 122 words. The ones above this are clearly corrupted and will be dropped in cleaning.


## 5. Episode Coverage


In [9]:
episodes = pd.read_csv("../data/raw_data/simpsons_episodes.csv")

print(
    f"Script lines: episode_id range {df['episode_id'].min()}-{df['episode_id'].max()}, "
    f"{df['episode_id'].nunique()} unique episodes"
)
print(
    f"Episodes file: {len(episodes)} episodes, seasons {episodes['season'].min()}-{episodes['season'].max()}"
)

# What seasons do the scripts cover?
script_episodes = episodes[episodes["id"].isin(df["episode_id"].unique())]
print(
    f"\nScripts cover seasons: {script_episodes['season'].min()} to {script_episodes['season'].max()}"
)

Script lines: episode_id range 1-568, 564 unique episodes
Episodes file: 600 episodes, seasons 1-28

Scripts cover seasons: 1 to 26


In [10]:
# Episodes per season in the script data
eps_per_season = script_episodes.groupby("season").size()
print("Episodes per season (in script data):")
print(eps_per_season.to_string())

Episodes per season (in script data):
season
1     13
2     22
3     24
4     22
5     22
6     25
7     25
8     25
9     25
10    23
11    22
12    21
13    22
14    22
15    22
16    21
17    22
18    22
19    20
20    19
21    22
22    22
23    22
24    22
25    21
26    16


The script data covers seasons 1–26 only. Season 26 is incomplete (16 episodes). Seasons 27–28 from the episodes file have no script data available.


## 7. Summary of Findings

Issues identified that will be addressed in [02_cleaning.ipynb](./02_cleaning.ipynb):

1. **Non-speaking lines (26,159 rows):** Stage directions and scene descriptions with no dialogue. We will filter to `speaking_line == "true"` only
2. **2 rows with missing character names:** Malformed entries with no speaker attribution. We will drop these.
3. **15 rows with corrupted columns:** CSV parsing errors from unescaped quotes causing non-numeric `word_count`. We will drop these.
4. **8 rows with outlier `word_count`:** Numeric but absurdly high values (108K–1.15M) from the same CSV parsing issue — timestamp values in the wrong column. We will drop rows with `word_count > 200`.
5. **Unused columns:** `id`, `raw_text`, `timestamp_in_ms`, `character_id`, `location_id`, `raw_location_text`, `normalized_text`. We will drop these.
6. **Missing season/episode metadata:** Join with `simpsons_episodes.csv` to add `season`, `number_in_season`, `title`, `imdb_rating`
7. **No sentence count:** Derive `sentence_count` by splitting on sentence-ending punctuation (needed for assignment question 5)
